# **Data Preprocessing**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

: 

In [ ]:
data = pd.read_csv('/content/video-game-sales.csv')
data.head()

In [ ]:
data.shape

In [ ]:
top_platforms = data['Platform'].value_counts()
top_platforms[top_platforms > 200]

In [ ]:
data = data[(data['Platform'] == 'PS2') | (data['Platform'] == 'PS3') | (data['Platform'] == 'PS4') |
            (data['Platform'] == 'X360') | (data['Platform'] == 'XOne') | (data['Platform'] == 'Wii') |
            (data['Platform'] == 'WiiU') | (data['Platform'] == 'PC')]

print('DataFrame Shape: ', data.shape)
data['Platform'].value_counts()

In [ ]:
# Missing Values Ratio
missing_ratio = (data.isnull().sum() / len(data)) * 100
missing_ratio[missing_ratio != 0].sort_values(ascending=False).head(16)

In [ ]:
# Droping high missing ratio columns
data.dropna(subset=['Critic_Score'], inplace=True)
missing_ratio = (data.isnull().sum() / len(data)) * 100
missing_ratio[missing_ratio != 0].sort_values(ascending=False).head(16)

In [ ]:
# Handling Missing Values
data['User_Count'] = data['User_Count'].fillna(data['User_Count'].mean())
data['Year_of_Release'] = data['Year_of_Release'].fillna(data['Year_of_Release'].median())
data['Rating'] = data['Rating'].fillna(data['Rating'].mode()[0])
data['Developer'] = data['Developer'].fillna(data['Developer'].mode()[0])
data['Publisher'] = data['Publisher'].fillna(data['Publisher'].mode()[0])

In [ ]:
data['User_Score'] = data['User_Score'].replace('tbd', np.nan)
data['User_Score'] = pd.to_numeric(data['User_Score'])
data['User_Score'] = data['User_Score'].fillna(data['User_Score'].median())

In [ ]:
missing_ratio = (data.isnull().sum() / len(data)) * 100
missing_ratio[missing_ratio != 0].sort_values(ascending=False).head(16)

In [ ]:
# Saving the Cleaned Data to a CSV File
data.to_csv('video-game-sales-cleaned.csv')

In [ ]:
# Label-Encoding
from sklearn.preprocessing import LabelEncoder

encoders = {}
categorical_columns = ['Platform', 'Genre', 'Rating']
for col in categorical_columns:
  encoder = LabelEncoder()
  data[col] = encoder.fit_transform(data[col])
  encoders[col] = encoder

data.head()

In [ ]:
# Dropping Irrelavent and Unncessary Features
cols = ['Name', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Developer']
data = data.drop(columns=cols, axis=1)
data.head()

In [ ]:
# Feature Engineering
data = data.rename(columns={'Year_of_Release':'Game_Age'})
data['Game_Age'] = 2025 - data['Game_Age']
data.head()

In [ ]:
plt.scatter(data['Critic_Score'], data['Global_Sales'])
plt.xlabel('Critic Score', fontsize=13)
plt.ylabel('Global Sales', fontsize=13)
plt.show()

In [ ]:
# Z-Score Normalization
from sklearn.preprocessing import StandardScaler

numeric_cols = ['Game_Age', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count']
scaler = StandardScaler()

data[numeric_cols] = scaler.fit_transform(data[numeric_cols])

In [ ]:
data.head()

In [ ]:
# Checking the Skewness of Global Sales
sns.histplot(data['Global_Sales'], bins=50, kde=True)
plt.show()

In [ ]:
data['Global_Sales'] = np.log1p(data['Global_Sales'])

In [ ]:
sns.histplot(data['Global_Sales'], bins=50, kde=True)
plt.show()

# **Model Training and Evaluation**

In [ ]:
# Train-Test Split
X = data.drop(columns=['Global_Sales'], axis=1)
y = data['Global_Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print('Training Examples: ', X_train.shape)
print('Test Examples: ', X_test.shape)

In [ ]:
def plot_regression_results(y_test, y_pred, model_name="Model"):

    plt.scatter(y_test, y_pred, alpha=0.6, edgecolor="k")
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{model_name} - Predicted vs Actual")

    plt.tight_layout()
    plt.show()

In [ ]:
models = {'Model':[], 'R2_Score':[], 'RMSE':[]}

### **K-Nearest Neighbors Regressor**

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

k_values = range(1, 31)
cv_scores = []
for k in k_values:
  knn = KNeighborsRegressor(n_neighbors=k)
  scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='r2')
  cv_scores.append(scores.mean())

optimal_k = k_values[np.argmax(cv_scores)]
print('Optimal K-Neighbors = ', optimal_k)

In [ ]:
plt.plot(k_values[:20], cv_scores[:20])
plt.xlabel('Number of Neighbors (K)')
plt.xticks(k_values[:20])
plt.ylabel('R2 Score')
plt.title('Cross-Validation Scores for Different Values of K')
plt.show()

In [ ]:
knn = KNeighborsRegressor(n_neighbors=optimal_k)
knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Root Mean Squared Error: ', rmse)
print('R2 Score: ', r2)

In [ ]:
models['Model'].append('KNN Regression')
models['R2_Score'].append(r2)
models['RMSE'].append(rmse)

In [ ]:
plot_regression_results(y_test, y_pred, model_name='KNN Regressor')

### **Linear Regression**

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

In [ ]:
y_pred = lin_reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Root Mean Squared Error: ', rmse)
print('R2 Score: ', r2)

In [ ]:
models['Model'].append('Linear Regression')
models['R2_Score'].append(r2)
models['RMSE'].append(rmse)

In [ ]:
plot_regression_results(y_test, y_pred, model_name='Linear Regression')

### **Support Vector Regressor**

In [ ]:
from sklearn.svm import SVR

param_grid = {'C': [0.01, 0.1, 1, 10],
               'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
               'kernel': ['rbf']}

grid = GridSearchCV(SVR(), param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)

svr = grid.best_estimator_

print('Best Parameters: ', grid.best_params_)

In [ ]:
y_pred = svr.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Root Mean Squared Error: ', rmse)
print('R2 Score: ', r2)

In [ ]:
models['Model'].append('Support Vector Regressor')
models['R2_Score'].append(r2)
models['RMSE'].append(rmse)

In [ ]:
plot_regression_results(y_test, y_pred, model_name='Support Vector Regressor')

### **Random Forest Regressor**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid = {'n_estimators': [100, 200, 300],
              'max_depth': [None, 10, 20],
              'min_samples_split': [2, 5, 10]}

grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

rf_reg = grid.best_estimator_

print('Best Parameters: ', grid.best_params_)

In [ ]:
y_pred = rf_reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Root Mean Squared Error: ', rmse)
print('R2 Score: ', r2)

In [ ]:
models['Model'].append('Random Forest Regressor')
models['R2_Score'].append(r2)
models['RMSE'].append(rmse)

In [ ]:
plot_regression_results(y_test, y_pred, model_name='Random Forest Regressor')

In [ ]:
importances = rf_reg.feature_importances_

feat_df = pd.DataFrame({'Feature': X.columns,'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

sns.barplot(data=feat_df, x='Importance', y='Feature',
    hue='Feature', palette='viridis')

plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## **Final Results**

In [ ]:
model_scores = pd.DataFrame(models)
model_scores = model_scores.sort_values(by='R2_Score', ascending=False).reset_index().drop(columns='index', axis=1)
model_scores

In [ ]:
plt.bar(model_scores['Model'], model_scores['R2_Score'], width=0.3)
plt.xlabel('Model', fontsize=14)
plt.xticks(rotation=20)
plt.ylabel('R2 Score', fontsize=14)
plt.title('Comparision of Different Models based on R2 Score', fontsize=15)
plt.show()

# **Prediting Video Game Sales for User Input**

In [ ]:
# Function to take user input and preprocess it
def preprocess_user_input(platform, year, genre, critic_score, critic_count, user_score, user_count, rating):
    # Create a DataFrame from user input
    user_data = pd.DataFrame({
        'Platform': [platform],
        'Game_Age': [2025 - year],
        'Genre': [genre],
        'Critic_Score': [critic_score],
        'Critic_Count': [critic_count],
        'User_Score': [user_score],
        'User_Count': [user_count],
        'Rating': [rating]
    })


    categorical_cols = ['Platform', 'Genre', 'Rating']
    for col in categorical_cols:
        if col in encoders:
            user_data[col] = encoders[col].transform(user_data[col])
        else:
            print(f"Warning: Encoder for {col} not found. Skipping encoding.")


    numeric_cols = ['Game_Age', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count']
    user_data[numeric_cols] = scaler.transform(user_data[numeric_cols])

    return user_data

# Function to predict global sales
def predict_global_sales(user_data, model):
    # Make prediction
    predicted_log_sales = model.predict(user_data)
    # Inverse transform the log scale prediction
    predicted_sales = np.expm1(predicted_log_sales)
    return predicted_sales[0]

# Get user input
print("Please enter the following details for the video game:")

unique_platforms = encoders['Platform'].classes_
unique_genres = encoders['Genre'].classes_
unique_ratings = encoders['Rating'].classes_


# Display options for categorical features
print(f"Available Platforms: {list(unique_platforms)}")
print(f"Available Genres: {list(unique_genres)}")
print(f"Available Ratings: {list(unique_ratings)}")


platform = input("Platform: ")
year = int(input("Year of Release: "))
genre = input("Genre: ")
critic_score = float(input("Critic Score (0-100): "))
critic_count = float(input("Critic Count: "))
user_score = float(input("User Score (0-10): "))
user_count = float(input("User Count: "))
rating = input("Rating: ")


# Preprocess user input
user_input_processed = preprocess_user_input(platform, year, genre, critic_score, critic_count, user_score, user_count, rating)

# Predict global sales using the best model (Random Forest Regressor)
predicted_sales = predict_global_sales(user_input_processed, rf_reg)
print(f"\nPredicted Global Sales: {predicted_sales:.2f} million units")